
# GPT & BERT  

This notebook demonstrates **practical programs** to understand the concepts behind **GPT** and **BERT** models.  

We will use the **Hugging Face Transformers** library with  backend.

### Learning Goals
- Understand the difference between **GPT (Generative)** and **BERT (Understanding)** models.
- Explore **text generation, classification, masked language modeling, question answering, and fine-tuning**.
- Learn about **tokenization, self-attention, and transfer learning** in practice.


# GPT models- OpenAI

Before running the notebook, follow these steps:

### 1. Install the OpenAI Python library




In [1]:
pip install openai


[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install transformers tensorflow datasets pandas -q


[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Get an API key

1. Sign up at [OpenAI's platform](https://platform.openai.com/)  
2. Generate an API key from your **API Keys Dashboard**  
3. Set it as an environment variable or use it directly in your code:

```bash
export OPENAI_API_KEY="your_api_key_here"

```

## 3. Important considerations

- GPT-4 API access may require a **paid account**  
- Be aware of **API costs** and set usage limits  
- `gpt-4-vision-preview` may have additional requirements  



## 1. Basic Text Generation with GPT-2

GPT is **autoregressive** → predicts the **next token** given previous tokens.  
We use GPT-2 for a simple text generation example with **Pytorch**.


In [3]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

def gpt2_text_generation(prompt, max_length=50):
    try:
        # Load pre-trained model and tokenizer
        model = GPT2LMHeadModel.from_pretrained('gpt2')
        tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

        # Set padding token (this fixes one warning)
        tokenizer.pad_token = tokenizer.eos_token

        # Encode the input prompt with explicit truncation
        input_ids = tokenizer.encode(
            prompt,
            return_tensors='pt',
            truncation=True,  # Explicit truncation
            max_length=512    # Set a reasonable max length for input
        )

        # Generate text - use max_new_tokens instead of max_length
        output = model.generate(
            input_ids,
            max_new_tokens=max_length,  # Use max_new_tokens instead
            num_return_sequences=1,
            no_repeat_ngram_size=2,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

        # Decode and return the generated text
        generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
        return generated_text

    except Exception as e:
        return f"Error generating text: {str(e)}"

# Example usage
prompt = "The future of artificial intelligence"
result = gpt2_text_generation(prompt)
print("Generated text:", result)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|████████████████████████████████████████████████████████████| 148/148 [00:00<00:00, 31106.28it/s]


Generated text: The future of artificial intelligence will be determined by the next generation of robots, but what's clear is that the technology is going to be better than the past.

"The key is for everybody to do it," says David Koopman, a professor at the University


In [4]:
import openai
import os

# Set your OpenAI API key
openai.api_key = os.getenv("OPENAI_API_KEY")  # Recommended: set as environment variable
# Alternatively: openai.api_key = "your-api-key-here"

def gpt4_chat_completion(prompt, model="gpt-4", max_tokens=500, temperature=0.7):
    """
    Basic function to get completion from GPT-4
    """
    try:
        response = openai.ChatCompletion.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=max_tokens,
            temperature=temperature,
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"An error occurred: {str(e)}"

# Example usage
prompt = "Explain the concept of quantum computing in simple terms."
response = gpt4_chat_completion(prompt)
print("GPT-4 Response:")
print(response)

GPT-4 Response:
An error occurred: 

You tried to access openai.ChatCompletion, but this is no longer supported in openai>=1.0.0 - see the README at https://github.com/openai/openai-python for the API.

You can run `openai migrate` to automatically upgrade your codebase to use the 1.0.0 interface. 

Alternatively, you can pin your installation to the old version, e.g. `pip install openai==0.28`

A detailed migration guide is available here: https://github.com/openai/openai-python/discussions/742




## 2. Text Classification with BERT

BERT is **autoencoding** → learns bidirectional context by predicting masked tokens.  
Here, we use **BERT for sentiment classification** with Pytorch.


In [5]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch
from torch.nn.functional import softmax

def bert_text_classification(text, model_name='bert-base-uncased'):
    """
    Perform text classification using pre-trained BERT
    """
    try:
        # Load pre-trained model and tokenizer
        tokenizer = BertTokenizer.from_pretrained(model_name)
        model = BertForSequenceClassification.from_pretrained(model_name)

        # Tokenize input text
        inputs = tokenizer(
            text,
            return_tensors='pt',
            truncation=True,
            padding=True,
            max_length=512
        )

        # Get model prediction
        with torch.no_grad():
            outputs = model(**inputs)

        # Process output
        logits = outputs.logits
        probabilities = softmax(logits, dim=1)
        predicted_class = torch.argmax(probabilities, dim=1).item()

        return predicted_class, probabilities.tolist()[0]

    except Exception as e:
        print(f"Error: {str(e)}")
        return None, None

# Example usage
text = "This movie was absolutely fantastic! The acting was superb."
class_id, probabilities = bert_text_classification(text)
print(f"Predicted class: {class_id}")
print(f"Probabilities: {probabilities}")

Loading weights: 100%|████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 27115.41it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were 

Predicted class: 0
Probabilities: [0.5219952464103699, 0.47800472378730774]



## 3. Question Answering with BERT

BERT can extract answers from context passages, fine-tuned on **SQuAD dataset**.


In [6]:
import torch
from transformers import BertForQuestionAnswering, BertTokenizer

def bert_question_answering(question, context):
    # Load tokenizer and model
    tokenizer = BertTokenizer.from_pretrained('bert-large-uncased-whole-word-masking-finetuned-squad')
    model = BertForQuestionAnswering.from_pretrained('bert-large-uncased-whole-word-masking-finetuned-squad')

    # Tokenize inputs
    inputs = tokenizer(question, context, return_tensors='pt', truncation=True, padding=True, max_length=512)

    # Get model outputs
    with torch.no_grad():
        outputs = model(**inputs)

    # Get start and end positions
    start_logits = outputs.start_logits
    end_logits = outputs.end_logits

    # Find the most likely start and end positions
    start_idx = torch.argmax(start_logits, dim=1).item()
    end_idx = torch.argmax(end_logits, dim=1).item() + 1

    # Extract answer tokens
    answer_tokens = inputs['input_ids'][0][start_idx:end_idx]

    # Decode the answer
    return tokenizer.decode(answer_tokens, skip_special_tokens=True)

context = """
The Transformers library is developed by Hugging Face. It provides general-purpose architectures
for Natural Language Understanding and Natural Language Generation.
"""

question = "Who develops the Transformers library?"
print("Answer:", bert_question_answering(question, context))

Loading weights: 100%|████████████████████████████████████████████████████████████| 391/391 [00:00<00:00, 21355.48it/s]
[transformers] BertForQuestionAnswering LOAD REPORT from: bert-large-uncased-whole-word-masking-finetuned-squad
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Answer: hugging face


## Other Models
## **Qwen** - The Developer/Organization
- **Developed by Alibaba Group** (formerly known as "Qianwen")
- One of China's leading AI research groups

## **Qwen2** - The Model Series
- Second generation of the Qwen model family
- Improved architecture and training compared to Qwen1
- Better performance across various benchmarks

## **7B** - The Model Size
- **7 Billion parameters**
- This is the number of trainable weights in the neural network
- A mid-sized model in the Qwen2 family (others include 0.5B, 1.5B, 72B versions)
- Requires about **14GB of GPU memory** for inference (in half-precision)

## **Instruct** - The Model Type
- Fine-tuned for instruction following
- Optimized for conversational and task-oriented interactions

In [1]:
pip install -U accelerate transformers torch


[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Use a valid Qwen model name
model_name = "Qwen/Qwen2-7B-Instruct"  # Changed to a valid, smaller model

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",  # Changed from dtype to torch_dtype
    device_map="auto",
)

# prepare the model input
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "user", "content": prompt},
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512,  # Reduced from 16384 to a more reasonable number
)

# Extract only the generated part (excluding the input)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]

content = tokenizer.decode(output_ids, skip_special_tokens=True)

print("Generated content:", content)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 4 files:   0%|                                                                        | 0/4 [3:20:44<?, ?it/s]


RuntimeError: Task error: File reconstruction error: CAS Client Error: Request middleware error: error sending request for url (https://us.aws.cdn.hf.co/xorbs/default/b7b985e17e15d2250a80c689e303132c5e83c50d0a857c1adf71d74f02cf69a7?user_id=public&repo_id=665ee74789b4fd787a568664&X-Xet-Session-Id=01a13ec7-d3de-7cc1-a00f-617624618b37&xip=hyW8QqcoKlo&Expires=1792058247&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hvcmJzL2RlZmF1bHQvYjdiOTg1ZTE3ZTE1ZDIyNTBhODBjNjg5ZTMwMzEzMmM1ZTgzYzUwZDBhODU3YzFhZGY3MWQ3NGYwMmNmNjlhN1xcP3VzZXJfaWQ9cHVibGljJnJlcG9faWQ9NjY1ZWU3NDc4OWI0ZmQ3ODdhNTY4NjY0JlgtWGV0LVNlc3Npb24tSWQ9MDFhMTNlYzctZDNkZS03Y2MxLWEwMGYtNjE3NjI0NjE4YjM3JnhpcD1oeVc4UXFjb0tsbyIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiRXBvY2hUaW1lIjoxNzkyMDU4MjQ3fSwiQnl0ZVJhbmdlIjp7IkV4cGVjdGVkSGVhZGVyIjoiYnl0ZXM9MC0xNDMxNzI0MSwxNDQwMTU0Mi0zOTY1MDkxOCwzOTc4ODA2MC01ODE3NzAyNyJ9fX1dfQ__&Signature=MEUCIADo7hl7ADlS8Y2D%7E9cdUNUi-QCDVAb7WQHF50IOMmjLAiEA1cSTXVsLOnv84LYcWrJZnZ8uzD74POx


# Summary

### GPT vs BERT
- **GPT** → Autoregressive, next-word prediction.
- **BERT** → Autoencoding, bidirectional context understanding.

### Key Concepts
- **Tokenization**: Converting text into tokens.
- **Self-Attention**: Mechanism to focus on different parts of input.
- **Fine-tuning**: Adapting pre-trained models to specific tasks.

### Ethical Considerations
- Biases in training data → unfair predictions.
- Responsible AI → avoid misuse in harmful applications.
